# fMRI ROI Analysis Pipeline (AAL Atlas)

This notebook performs:
- BIDS file discovery
- per-subject first-level GLM using fMRIPrep outputs
- ROI extraction with the AAL atlas
- contrast comparison across conditions
- group ROI statistics and visualization


In [ ]:
# PARAMETERS
bids_root = "/path/to/your/BIDS"
fmriprep_deriv = "/path/to/your/BIDS/derivatives/fmriprep"
output_dir = "./results_fmri_roi"
task_label = "learning"
subject_list = []  # leave empty to auto-discover
smoothing_fwhm = None
n_jobs = 1

import os, glob
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from nilearn import image, datasets
from nilearn.input_data import NiftiLabelsMasker
from nilearn.glm.first_level import FirstLevelModel
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(output_dir, exist_ok=True)


In [ ]:
def find_subjects(bids_root, fmriprep_deriv):
    subs = []
    for p in glob.glob(os.path.join(fmriprep_deriv, "sub-*")):
        if os.path.isdir(p):
            sub = os.path.basename(p).split('-', 1)[1]
            subs.append(sub)
    if len(subs) == 0:
        for p in glob.glob(os.path.join(bids_root, "sub-*")):
            if os.path.isdir(p):
                sub = os.path.basename(p).split('-', 1)[1]
                subs.append(sub)
    return sorted(list(set(subs)))

if len(subject_list) == 0:
    subject_list = find_subjects(bids_root, fmriprep_deriv)

print('Subjects detected:', subject_list)


In [ ]:
# Load AAL atlas
aal = datasets.fetch_atlas_aal()
atlas_img = aal['maps']
atlas_labels = aal['labels']

masker = NiftiLabelsMasker(labels_img=atlas_img, standardize=True)
roi_df = pd.DataFrame({'roi_index': range(len(atlas_labels)), 'roi_name': atlas_labels})
roi_df.to_csv(os.path.join(output_dir, 'roi_list.csv'), index=False)
print('AAL atlas loaded:', atlas_img)


In [ ]:
def find_subject_files(sub, task_label):
    bold_candidates = sorted(glob.glob(os.path.join(fmriprep_deriv, f'sub-{sub}', 'func', f'sub-{sub}_task-{task_label}_*space-MNI*_desc-preproc_bold.nii*')))
    events_file = os.path.join(bids_root, f'sub-{sub}', 'func', f'sub-{sub}_task-{task_label}_events.tsv')
    confounds = sorted(glob.glob(os.path.join(fmriprep_deriv, f'sub-{sub}', 'func', f'sub-{sub}_task-{task_label}_*confounds.tsv')))
    return bold_candidates, events_file, confounds


In [ ]:
subject_roi_values = {}
group_contrast_imgs = {}

for sub in subject_list:
    print(f'Processing subject {sub}...')
    bolds, events_file, confounds = find_subject_files(sub, task_label)
    if len(bolds) == 0 or not os.path.exists(events_file):
        print(f'  Skip {sub}: no preprocessed bold or events file found.')
        continue

    events_df = pd.read_csv(events_file, sep='\t')
    if 'trial_type' not in events_df.columns:
        print(f'  Skip {sub}: no trial_type in events file.')
        continue

    events_list = [events_df.copy() for _ in bolds]
    confounds_list = []
    for conf in confounds:
        try:
            cf = pd.read_csv(conf, sep='\t')
            motion_cols = [c for c in cf.columns if c.startswith('trans_') or c.startswith('rot_')]
            if 'framewise_displacement' in cf.columns:
                motion_cols += ['framewise_displacement']
            if len(motion_cols) > 0:
                confounds_list.append(cf[motion_cols])
            else:
                confounds_list.append(None)
        except Exception:
            confounds_list.append(None)

    while len(confounds_list) < len(bolds):
        confounds_list.append(None)

    try:
        first_level_model = FirstLevelModel(
            t_r=None,
            hrf_model='spm',
            smoothing_fwhm=smoothing_fwhm,
            noise_model='ar1',
            standardize=False,
            signal_scaling=0,
            n_jobs=n_jobs,
            verbose=0,
        )
        first_level_model = first_level_model.fit(bolds, events=events_list, confounds=confounds_list)
    except Exception as e:
        print(f'  GLM fit failed for {sub}: {e}')
        continue

    conds = sorted(events_df['trial_type'].dropna().unique().tolist())
    subject_roi_values[sub] = {}

    for cond in conds:
        try:
            effect_img = first_level_model.compute_contrast(cond, output_type='effect_size')
            sub_dir = os.path.join(output_dir, f'sub-{sub}')
            os.makedirs(sub_dir, exist_ok=True)
            out_path = os.path.join(sub_dir, f'sub-{sub}_task-{task_label}_contrast-{cond}_effect.nii.gz')
            effect_img.to_filename(out_path)
            roi_vals = masker.fit_transform(effect_img).ravel()
            subject_roi_values[sub][cond] = roi_vals
            group_contrast_imgs.setdefault(cond, []).append(out_path)
        except Exception as e:
            print(f'  Contrast {cond} failed for {sub}: {e}')
            continue

print('Finished per-subject processing')


In [ ]:
rows = []
for sub, cond_dict in subject_roi_values.items():
    for cond, vals in cond_dict.items():
        for roi_idx, val in enumerate(vals):
            rows.append({
                'subject': sub,
                'condition': cond,
                'roi_idx': roi_idx,
                'roi_name': atlas_labels[roi_idx],
                'value': float(val)
            })

df_roi_all = pd.DataFrame(rows)
df_roi_all.to_csv(os.path.join(output_dir, 'subject_roi_values.csv'), index=False)
df_roi_all.head()


In [ ]:
group_rows = []
for cond, img_paths in group_contrast_imgs.items():
    if len(img_paths) == 0:
        continue
    sub_vals = []
    for sub in subject_list:
        if sub in subject_roi_values and cond in subject_roi_values[sub]:
            sub_vals.append(subject_roi_values[sub][cond])
    if len(sub_vals) == 0:
        continue
    arr = np.vstack(sub_vals)
    mean_vals = arr.mean(axis=0)
    t_vals = []
    p_vals = []
    for i in range(arr.shape[1]):
        tval, pval = stats.ttest_1samp(arr[:, i], 0.0, nan_policy='omit')
        t_vals.append(float(tval))
        p_vals.append(float(pval))
    reject, p_fdr, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')

    for idx, roi_name in enumerate(atlas_labels):
        group_rows.append({
            'condition': cond,
            'roi_idx': idx,
            'roi_name': roi_name,
            'mean_effect': float(mean_vals[idx]),
            't': float(t_vals[idx]),
            'p_uncorrected': float(p_vals[idx]),
            'p_fdr': float(p_fdr[idx]),
            'significant_fdr': bool(reject[idx])
        })

group_stats = pd.DataFrame(group_rows)
group_stats.to_csv(os.path.join(output_dir, 'group_roi_stats.csv'), index=False)
group_stats.head()


In [ ]:
for cond, paths in group_contrast_imgs.items():
    if len(paths) == 0:
        continue
    mean_img = image.mean_img(paths)
    out_path = os.path.join(output_dir, f'group_task-{task_label}_contrast-{cond}_mean_effect.nii.gz')
    mean_img.to_filename(out_path)
    print('Saved group mean map:', out_path)


In [ ]:
if len(group_contrast_imgs.keys()) > 0:
    cond0 = list(group_contrast_imgs.keys())[0]
    gdf = group_stats[group_stats['condition'] == cond0].copy()
    top = gdf.sort_values('p_fdr').head(10)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=top, x='mean_effect', y='roi_name', palette='vlag')
    plt.title(f'Top ROIs for condition: {cond0}')
    plt.xlabel('Mean effect size')
    plt.ylabel('ROI')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{cond0}_top_rois.png'))
    plt.show()

    map_path = os.path.join(output_dir, f'group_task-{task_label}_contrast-{cond0}_mean_effect.nii.gz')
    if os.path.exists(map_path):
        stat_map = image.load_img(map_path)
        from nilearn import plotting
        display = plotting.plot_stat_map(stat_map, title=f'Group mean effect: {cond0}', threshold=None, display_mode='ortho')
        display.savefig(os.path.join(output_dir, f'{cond0}_stat_map.png'))
        display.close()


## Completed
The output is saved in the directory `results_fmri_roi`.
You can use the Streamlit app to inspect the ROI activation interactively.
